# D8 flow direction: follow the steepest descent

A DEM gives us an elevation for each cell. D8 chooses one of the eight neighbouring cells as the next step downhill.

This lab creates a small synthetic hill and draws the flow direction from each cell. Move the hill and change the overall tilt to see how the local directions change.

**You need:** Python, NumPy, Matplotlib and ipywidgets.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, interact

DIRECTIONS = [
    (0, 1, 1), (1, 1, 2), (1, 0, 4), (1, -1, 8),
    (0, -1, 16), (-1, -1, 32), (-1, 0, 64), (-1, 1, 128),
]


## 1. Build a synthetic elevation surface

The surface is a simple hill. Elevation is highest near its centre and decreases away from it. This makes the flow pattern easy to see.

In [ ]:
def make_dem(peak_x=4.0, peak_y=4.0, east_tilt=0.0):
    rows, cols = np.mgrid[0:9, 0:9]
    dem = -((cols - peak_x) ** 2 + (rows - peak_y) ** 2) + east_tilt * cols
    return dem

dem = make_dem()
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(dem, origin="upper", cmap="terrain")
fig.colorbar(image, ax=ax, label="Elevation (illustrative)")
ax.set(title="Synthetic DEM", xlabel="Column", ylabel="Row")
plt.show()

## 2. Apply the D8 rule

For each cell, calculate the drop divided by centre-to-centre distance to every valid neighbour. Choose the largest positive gradient.

This notebook uses a fixed tie order and leaves sinks/flats unresolved.

In [ ]:
def d8_codes(dem, cell_width=1.0, cell_height=1.0):
    rows, cols = dem.shape
    result = np.zeros((rows, cols), dtype=int)
    for r in range(rows):
        for c in range(cols):
            best_slope, best_code = 0.0, 0
            for dr, dc, code in DIRECTIONS:
                rr, cc = r + dr, c + dc
                if not (0 <= rr < rows and 0 <= cc < cols):
                    continue
                distance = np.hypot(dc * cell_width, dr * cell_height)
                slope = (dem[r, c] - dem[rr, cc]) / distance
                if slope > best_slope:
                    best_slope, best_code = slope, code
            result[r, c] = best_code
    return result

codes = d8_codes(dem)
print("Unique direction codes:", sorted(np.unique(codes).tolist()))

## 3. Draw the flow directions

Each arrow points from the cell centre toward its selected receiving neighbour. A code of 0 means no strictly lower neighbour was found in this simplified version.

In [ ]:
def draw_flow(peak_x=4.0, peak_y=4.0, east_tilt=0.0):
    surface = make_dem(peak_x, peak_y, east_tilt)
    codes = d8_codes(surface)
    offsets = {code: (dr, dc) for dr, dc, code in DIRECTIONS}

    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(surface, origin="upper", cmap="terrain")
    fig.colorbar(image, ax=ax, label="Elevation")
    for r in range(surface.shape[0]):
        for c in range(surface.shape[1]):
            code = codes[r, c]
            if code == 0:
                ax.scatter(c, r, marker="x", color="black", s=28)
                continue
            dr, dc = offsets[code]
            ax.arrow(c, r, dc * 0.34, dr * 0.34,
                     head_width=0.12, head_length=0.12,
                     length_includes_head=True, color="black")
    ax.set(title="D8 flow direction", xlabel="Column", ylabel="Row")
    ax.set_xticks(range(9))
    ax.set_yticks(range(9))
    plt.show()

interact(
    draw_flow,
    peak_x=FloatSlider(value=4.0, min=1, max=7, step=0.5, description="Peak x"),
    peak_y=FloatSlider(value=4.0, min=1, max=7, step=0.5, description="Peak y"),
    east_tilt=FloatSlider(value=0.0, min=-3, max=3, step=0.25, description="East tilt"),
);

## 4. Questions to investigate

1. Why do diagonal moves use a longer distance than orthogonal moves?
2. What happens to the flow pattern when you move the hill?
3. How does the east-west tilt change the directions?
4. Which cells have no strictly lower neighbour?
5. What extra processing would be needed to resolve flats and depressions?

## 5. Important limits

This is a local D8 demonstration, not a full hydrological conditioning workflow. It does not fill sinks, resolve flats, or route flow beyond the raster boundary. Real work needs a suitable DEM, cell dimensions, NoData policy, depression treatment and validation.

Continue with the [D8 lesson](../algorithms/hydrology/d8-flow-direction.md) and [tested Python implementation](../implementations/python/d8-flow-direction/README.md).